# ScavenPred and ACE-iPred — Reproducible statistical benchmarking

**Current final training-only forward-wrapper models vs archived external tools, 10 November 2026**

This is an *executable research notebook*, not a table of copied P-values. It uses original per-sequence observed labels, current models' predictions and probabilities, and archived third-party models' predictions. **All comparisons are performed on the same matched, similarity-screened peptides** (ScavenPred vs AOPxSVM: n=250; ACE-iPred vs UniDL4BioPep: n=248).

The workflow is: load and audit paired predictions → compute confusion matrices and classification metrics → calculate stratified bootstrap 95% CIs → exact paired McNemar tests → paired bootstrap differences in accuracy/MCC/ROC-AUC/PR-AUC → inspect discordant classifications → generate figures and CSV tables.

**Source limitation:** AnOxPePred's complete peptide-level predictions were not recovered; its earlier aggregate metrics (n=256) cannot be paired against the new n=250 ScavenPred cohort. Its P-values are **not** reused. Archived comparators were not rerun; their original saved predictions were matched by peptide sequence.

**Statistical caution:** The paired bootstrap P-values are approximate, depend on resampling and are unadjusted for multiple comparisons. Exact McNemar P-values test accuracy discordance, not MCC. PR-AUC is operationalized as average precision. These analyses do not imply causal or experimental validation.

To run: place this notebook in the same folder as the two bundled `*_paired_predictions.csv` files, then use **Run All**. Dependencies: Python, numpy, pandas, scipy, scikit-learn, matplotlib.

## 1. Load libraries and establish reproducibility

In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.metrics import (accuracy_score, matthews_corrcoef, roc_auc_score,
                             average_precision_score, recall_score, precision_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, precision_recall_curve)
import sklearn

SEED = 20261110
N_BOOT = 4000  # Number used for the archival paired benchmarking report
RNG = np.random.default_rng(SEED)
HERE = Path.cwd()
OUT = HERE / 'results'
OUT.mkdir(exist_ok=True)
print('Python sklearn:', sklearn.__version__)
print('Reproducibility seed:', SEED, 'bootstrap replicates:', N_BOOT)

## 2. Load sequence-matched comparator records

Each row contains the peptide sequence, observed activity class, current model binary prediction and probability, and comparator prediction and probability. The original provenance audit matched exact sequences and removed training-related test peptides. This notebook checks the paired files again.

In [ ]:
filenames = {
    'ScavenPred': 'ScavenPred_current_vs_AOPxSVM_paired_predictions.csv',
    'ACE-iPred': 'ACE-iPred_current_vs_UniDL4BioPep_paired_predictions.csv',
}
comparator_names = {'ScavenPred':'AOPxSVM', 'ACE-iPred':'UniDL4BioPep'}
paired = {}
for name, file in filenames.items():
    path = HERE/file
    if not path.exists():
        raise FileNotFoundError(f'Missing bundled matched-prediction file: {path}')
    df = pd.read_csv(path)
    required = {'sequence','truth','true','our_pred','our_prob','other_pred','other_prob',
                'excluded_related_to_train'}
    assert required <= set(df.columns), f'{name}: missing {required-set(df.columns)}'
    assert df.sequence.is_unique and df.sequence.notna().all()
    assert (~df.excluded_related_to_train.astype(bool)).all()
    assert df[['truth','true','our_pred','other_pred','our_prob','other_prob']].notna().all().all()
    assert (df.truth.astype(int)==df['true'].astype(int)).all(), 'Mismatched observed class labels'
    assert set(df.truth.unique())=={0,1}
    assert df[['our_pred','other_pred']].isin([0,1]).all().all()
    assert df[['our_prob','other_prob']].ge(0).all().all() and df[['our_prob','other_prob']].le(1).all().all()
    paired[name]=df
    print(f'{name} vs {comparator_names[name]}: n={len(df)}, pos={(df.truth==1).sum()}, neg={(df.truth==0).sum()}')
assert len(paired['ScavenPred'])==250 and len(paired['ACE-iPred'])==248


## 3. Calculate all point estimates from the underlying predictions

Sensitivity = TP/(TP+FN), specificity = TN/(TN+FP), MCC uses all four confusion matrix cells, ROC-AUC uses continuous positive-class scores, and PR-AUC is average precision. This recreates model metrics without relying on manuscript tables.

In [ ]:
def score_metrics(y, predicted, probability):
    y=np.asarray(y,dtype=int); predicted=np.asarray(predicted,dtype=int)
    probability=np.asarray(probability,dtype=float)
    tn,fp,fn,tp=confusion_matrix(y,predicted,labels=[0,1]).ravel()
    return {'Accuracy':accuracy_score(y,predicted),
            'Sensitivity':recall_score(y,predicted,zero_division=0),
            'Specificity':tn/(tn+fp) if tn+fp else np.nan,
            'Precision':precision_score(y,predicted,zero_division=0),
            'F1':f1_score(y,predicted,zero_division=0),
            'MCC':matthews_corrcoef(y,predicted),
            'ROC-AUC':roc_auc_score(y,probability),
            'PR-AUC':average_precision_score(y,probability),
            'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)}

metric_rows=[]
for task, df in paired.items():
    y=df.truth.astype(int).to_numpy()
    for label,pcol,probcol in [(task,'our_pred','our_prob'),
                               (comparator_names[task],'other_pred','other_prob')]:
        metric_rows.append({'Task':task,'Predictor':label,'n':len(y),
                            **score_metrics(y,df[pcol],df[probcol])})
metrics=pd.DataFrame(metric_rows)
display(metrics.round(4))
metrics.to_csv(OUT/'Matched_Independent_Performance.csv',index=False)
assert abs(metrics.loc[metrics.Predictor=='ScavenPred','MCC'].iloc[0] - 0.438924355585)<1e-9
assert abs(metrics.loc[metrics.Predictor=='ACE-iPred','MCC'].iloc[0] - 0.491398685358)<1e-9

## 4. Accuracy discordance and exact two-sided McNemar test

Discordance uses **correct vs incorrect on each same peptide**, not simply differences in predicted labels. If `b` denotes only the current model correct and `c` denotes only the comparator correct, the exact two-sided McNemar P-value is the binomial test of `min(b,c)` successes in `b+c` trials under probability 0.5.

In [ ]:
discordance_rows=[]
for task,df in paired.items():
    truth=df.truth.to_numpy(dtype=int)
    own=df.our_pred.to_numpy(dtype=int)==truth
    other=df.other_pred.to_numpy(dtype=int)==truth
    both=int(np.sum(own & other))
    b=int(np.sum(own & ~other))
    c=int(np.sum(~own & other))
    neither=int(np.sum(~own & ~other))
    p=binomtest(min(b,c),b+c,p=0.5,alternative='two-sided').pvalue if b+c else 1.0
    assert both+b+c+neither==len(df)
    discordance_rows.append({'Model':task,'Comparator':comparator_names[task], 'n':len(df),
                             'Both correct':both,'Current only correct (b)':b,
                             'Comparator only correct (c)':c,'Both incorrect':neither,
                             'McNemar exact P':p})
discordance=pd.DataFrame(discordance_rows)
display(discordance)
assert (discordance['Current only correct (b)'].to_numpy()==[44,47]).all()
assert (discordance['Comparator only correct (c)'].to_numpy()==[31,25]).all()
discordance.to_csv(OUT/'Exact_McNemar_Discordance.csv',index=False)

## 5. Stratified paired bootstrap: individual metrics and paired differences

Resample the **same peptide row indices for both predictors** within each true class; this preserves the original positive and negative cohort sizes and within-peptide covariance. For each replicate compute accuracy, MCC, ROC-AUC and average precision for each predictor, and calculate the difference *current minus comparator*. Report percentile 95% confidence intervals. An approximate two-sided P-value is twice the smaller empirical tail proportion around zero; it is not an exact test.

In [ ]:
PRIMARY_METRICS=['Accuracy','MCC','ROC-AUC','PR-AUC']
CI_METRICS=['Accuracy','Sensitivity','Specificity','Precision','F1','MCC','ROC-AUC','PR-AUC']

def stratified_resamples(y,n_resamples,rng):
    pos=np.flatnonzero(y==1); neg=np.flatnonzero(y==0)
    for _ in range(n_resamples):
        yield np.r_[rng.choice(pos,len(pos),replace=True),
                    rng.choice(neg,len(neg),replace=True)]

def paired_bootstrap(df,n_resamples=4000,seed=20261110):
    local_rng=np.random.default_rng(seed)
    y=df.truth.to_numpy(int)
    p1=df.our_pred.to_numpy(int); s1=df.our_prob.to_numpy(float)
    p2=df.other_pred.to_numpy(int); s2=df.other_prob.to_numpy(float)
    estimates1=score_metrics(y,p1,s1); estimates2=score_metrics(y,p2,s2)
    original1={k:[] for k in CI_METRICS}
    original2={k:[] for k in CI_METRICS}
    differences={k:[] for k in PRIMARY_METRICS}
    for ix in stratified_resamples(y,n_resamples,local_rng):
        a=score_metrics(y[ix],p1[ix],s1[ix]); b=score_metrics(y[ix],p2[ix],s2[ix])
        for key in CI_METRICS:
            original1[key].append(a[key]);original2[key].append(b[key])
        for key in PRIMARY_METRICS: differences[key].append(a[key]-b[key])
    return estimates1,estimates2,original1,original2,differences

bootstrap_results={}
for name,df in paired.items():
    bootstrap_results[name]=paired_bootstrap(df,N_BOOT,SEED)
print('Paired, class-stratified resampling completed.')

## 6. Confidence intervals and paired difference tests

In [ ]:
ci_rows=[]; difference_rows=[]
for task,(e1,e2,reps1,reps2,diffs) in bootstrap_results.items():
    for model,point,reps in ((task,e1,reps1),(comparator_names[task],e2,reps2)):
        for key in CI_METRICS:
            low,high=np.percentile(reps[key],[2.5,97.5])
            ci_rows.append({'Task':task,'Predictor':model,'Metric':key,
                            'Estimate':point[key],'CI_low':low,'CI_high':high})
    for metric in PRIMARY_METRICS:
        arr=np.asarray(diffs[metric]); low,high=np.percentile(arr,[2.5,97.5])
        tails=2*min(np.mean(arr<=0), np.mean(arr>=0))
        difference_rows.append({'Model':task,'Comparator':comparator_names[task],
            'n':len(paired[task]),'Metric':metric,'Current':e1[metric],
            'Comparator_score':e2[metric],'Difference':e1[metric]-e2[metric],
            'CI_low':low,'CI_high':high,'Paired_bootstrap_P_approx':min(1.,tails),
            'Exact_McNemar_P_for_accuracy':float(discordance.loc[discordance.Model==task,'McNemar exact P'].iloc[0]) if metric=='Accuracy' else np.nan})
ci_table=pd.DataFrame(ci_rows)
differences_table=pd.DataFrame(difference_rows)
display(ci_table.round(4))
display(differences_table.round(4))
ci_table.to_csv(OUT/'Bootstrap_Metric_95CI.csv',index=False)
differences_table.to_csv(OUT/'Paired_Bootstrap_Comparisons.csv',index=False)

## 7. Figures: confusion matrices, ROC/PR curves and confidence intervals

Every figure is generated from the matched, independent peptide predictions. The paired difference plot includes a vertical zero-reference line.

In [ ]:
for task,df in paired.items():
    y=df.truth.to_numpy(int)
    fig, axes=plt.subplots(1,2,figsize=(8.5,3.4),constrained_layout=True)
    for ax,who,col in [(axes[0],task,'our_pred'),(axes[1],comparator_names[task],'other_pred')]:
        ConfusionMatrixDisplay.from_predictions(y,df[col].to_numpy(int),
            labels=[0,1],ax=ax,colorbar=False,cmap='Blues')
        ax.set_title(who)
    fig.savefig(OUT/(task+'_confusion_matrices.png'),dpi=240)
    plt.show()

    fig,axes=plt.subplots(1,2,figsize=(9,3.6),constrained_layout=True)
    for who,col in [(task,'our_prob'),(comparator_names[task],'other_prob')]:
        scores=df[col].to_numpy(float)
        fpr,tpr,_=roc_curve(y,scores)
        precision,recall,_=precision_recall_curve(y,scores)
        axes[0].plot(fpr,tpr,label=f'{who}: {roc_auc_score(y,scores):.3f}')
        axes[1].plot(recall,precision,label=f'{who}: {average_precision_score(y,scores):.3f}')
    axes[0].plot([0,1],[0,1],'--',color='gray',alpha=.6)
    axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title=task+' ROC')
    axes[1].set(xlabel='Recall',ylabel='Precision',title=task+' Precision–recall')
    for ax in axes:ax.legend(fontsize=8)
    fig.savefig(OUT/(task+'_ROC_PR.png'),dpi=240)
    plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.1),constrained_layout=True)
for ax,task in zip(axes,paired):
    frame=differences_table[differences_table.Model==task].copy()
    y=np.arange(len(frame))[::-1]
    ax.errorbar(frame.Difference,y,
        xerr=[frame.Difference-frame.CI_low,frame.CI_high-frame.Difference],
        fmt='o',capsize=4)
    ax.axvline(0,linestyle='--',color='gray',linewidth=1)
    ax.set_yticks(y,frame.Metric)
    ax.set_xlabel('Current − comparator (paired difference)')
    ax.set_title(f'{task} vs {comparator_names[task]}')
fig.savefig(OUT/'Paired_Difference_95CI.png',dpi=260)
fig.savefig(OUT/'Paired_Difference_95CI.pdf')
plt.show()

## 8. Identify the exact discordant peptides

This audit enables manual review of which peptides one predictor classified correctly while the other did not. It is also the source of the ACE-iPred 47-versus-25 statement.

In [ ]:
for task,df in paired.items():
    tmp=df[['sequence','truth','our_pred','our_prob','other_pred','other_prob']].copy()
    a=(tmp.our_pred.astype(int)==tmp.truth.astype(int))
    b=(tmp.other_pred.astype(int)==tmp.truth.astype(int))
    tmp['correctness_category']=np.select(
        [a&b,a&~b,~a&b],['Both correct','Current only correct','Comparator only correct'],
        default='Both incorrect')
    path=OUT/(task+'_per_peptide_correctness.csv')
    tmp.to_csv(path,index=False)
    print(f'{task}: category frequencies')
    display(tmp.correctness_category.value_counts().to_frame())
    display(tmp.loc[tmp.correctness_category=='Current only correct'].head(10))

## 9. Comparison to historical model versions (optional, separate experiment)

The **current vs previous version of the same tools** is an entirely different comparison from **current vs AOPxSVM/UniDL4BioPep**. The earlier version was assessed with paired 95% bootstrap and McNemar; the summary below is archival. To recompute those tests in this notebook, one must supply the original *previous-version per-sequence predictions*, match by sequence and use the same functions above. Never compare unrelated cohort totals.

In [ ]:
historical_summary = pd.DataFrame([
    {'Model':'ScavenPred','Historical_model_MCC':0.380757,'Current_model_MCC':0.438924,
     'MCC_difference':0.058167,'95CI_low':-0.042935,'95CI_high':0.154851,'Exact_McNemar_P':0.332306},
    {'Model':'ACE-iPred','Historical_model_MCC':0.419010,'Current_model_MCC':0.491399,
     'MCC_difference':0.072389,'95CI_low':-0.048015,'95CI_high':0.188788,'Exact_McNemar_P':0.331750}
])
display(historical_summary)
print('These archival summary values are NOT recomputed from the two third-party comparison CSVs.')

## 10. How to report the statistics responsibly

- **ScavenPred vs AOPxSVM:** evaluate the *250* paired peptides. The primary classification conclusion should reflect the exact McNemar test (accuracy) and paired MCC CI independently. Do not claim significant accuracy superiority if the exact McNemar test is not significant.
- **ACE-iPred vs UniDL4BioPep:** evaluate the *248* paired peptides. The current ACE model alone correctly classified 47 versus 25 by UniDL4BioPep alone; the correct P-value is calculated directly from those discordant counts.
- **AnOxPePred:** no current row-matched verified prediction file exists; original n=256 aggregate values and historical P-values are not transferable to the current n=250 cohort.
- **Bootstrap P-values:** exploratory approximate two-sided estimates, not exact permutation tests. Report how many resamples, seed and whether multiplicity was controlled.
- **External independence:** the current models did not use the independent test activity labels for feature selection or fitting. Sequence similarity filtering was based on sequence relationships, not outcomes.
- **Data provenance:** this notebook starts from the archived, already-matched per-sequence prediction files. It does **not** call third-party websites or rerun those predictors.

Outputs appear in the `results/` directory.